# Tabular GAN через SDV

Минимальный скелет ноутбука: загрузить таблицу в `df`, убрать лишние колонки, обучить `CTGANSynthesizer`, получить `synthetic_df`.

## Зависимости

In [1]:
from pathlib import Path

import pandas as pd
from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer

## Загрузка таблицы

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TABLE_NAME = "table"
TABLE_PATH = PROJECT_ROOT / "research/data/incident_base_llm_разметка.xlsx"

In [3]:
if TABLE_PATH.suffix.lower() in {".xlsx", ".xls"}:
    df = pd.read_excel(TABLE_PATH)
else:
    df = pd.read_csv(TABLE_PATH)

display(df.head())
df.shape

,Дата,Время,Адрес,Пом.,Подкатегория,Описание,Тип инцидента
0,2026-03-15,15:04,"г Санкт-Петербург, п Парголово, ул Валерия Гав...",NaN,Лифт,4ПАР НЕ РАБ-Т 1ПАССАЖ ЛИФТ И 1-Н ГРУЗОВОЙ ЛИФ...,Лифты
1,2026-03-15,12:42,"г Санкт-Петербург, п Парголово, ул Валерия Гав...",NaN,Лифт,4 ПАР НЕ РАБОТАЮТ 3-ЛИФТА ИЗ 4,Лифты
2,2026-03-15,12:33,"г Санкт-Петербург, ул Ивинская, д. 15 стр. 1",NaN,Лифт,1 ПАР НЕ РАБОТАЕТ ГР/ЛИФТ,Лифты
3,2026-03-15,12:27,"г Санкт-Петербург, п Парголово, ул Валерия Гав...",NaN,Лифт,7 ПАР ГР ЛИФТ НЕ РАБОТАЕТ,Лифты
4,2026-03-15,12:15,"г Санкт-Петербург, п Парголово, ул Фёдора Абра...",NaN,Лифт,6 ПАР ГР ЛИФТ НЕ РАБОТАЕТ,Лифты


(458, 7)

## Подготовка

In [4]:
DROP_COLUMNS = ["Описание"]

df_model = df.drop(columns=DROP_COLUMNS, errors="ignore").copy()
df_model.shape

(458, 6)

## Конфиг обучения

In [5]:
CTGAN_PARAMS = {
    "epochs": 100,
    "batch_size": 500,
    "pac": 10,
    "embedding_dim": 128,
    "generator_dim": (256, 256),
    "discriminator_dim": (256, 256),
    "generator_lr": 2e-4,
    "discriminator_lr": 2e-4,
    "discriminator_steps": 1,
    "enforce_min_max_values": True,
    "enforce_rounding": True,
    "enable_gpu": True,
    "verbose": True,
}

## Обучение

In [6]:
metadata = Metadata.detect_from_dataframe(
    data=df_model,
    table_name=TABLE_NAME,
)

synthesizer = CTGANSynthesizer(
    metadata=metadata,
    **CTGAN_PARAMS,
)
synthesizer.fit(df_model)

/home/fedor/Projects/building_maintenance_agents/.venv/lib/python3.12/site-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (3.06) | Discrim. (0.18): 100%|██████████| 100/100 [00:03<00:00, 26.89it/s]


## Генерация

In [7]:
SYNTHETIC_ROWS = len(df_model)

synthetic_df = synthesizer.sample(num_rows=SYNTHETIC_ROWS)
display(synthetic_df.head())
synthetic_df.shape

,Дата,Время,Адрес,Пом.,Подкатегория,Тип инцидента
0,1970-01-21,12:32,"г Санкт-Петербург, п Парголово, ул Шишкина, д....",NaN,Протечка,Лифты
1,1970-01-21,22:45,"г Санкт-Петербург, п Парголово, ул Фёдора Абра...",NaN,Лифт,Лифты
2,1970-01-21,19:15,"г Санкт-Петербург, п Парголово, ул Фёдора Абра...",NaN,Лифт,Стояк ГВС
3,1970-01-21,11:38,"г Санкт-Петербург, п Парголово, ул Николая Руб...",NaN,Электроснабжение,Лифты
4,1970-01-21,12:12,"г Санкт-Петербург, п Парголово, ул Фёдора Абра...",421,Протечка,Лифты


(458, 6)